# ACTG 320 - ACTG 175

The following example comes from the AIDS Clinical Trial Group (ACTG) 320 and 175 trials. The ACTG 320 was a randomized trial comparing 3-drug to 2-drug antiretroviral therapy (ART) among persons with HIV. The ACTG 175 was a randomized trial comparing 2-drug to 1-drug ART. Here, we will compare 3-drug to 1-drug ART in the ACTG 320 source population. The outcome of interest will be the composite outcome of AIDS, death, or $\ge 50$% decline in CD4 at 200 days post-randomization (note: the time-to-event aspect of the data is ignored and those right censored prior to 200 days simply have their outcome set to be missing).

Here, we will illustrate three different estimators: a bridge inverse probability weighting estimator, a bridge g-computation estimator, and a bridge augmented inverse probability weighting estimator. 

## Setup

In [1]:
import numpy as np
import pandas as pd
import delicatessen as deli
from delicatessen.utilities import spline

import cantilever
from cantilever.estimators.point import BridgeIPW, BridgeGComputation, BridgeAIPW

print("Versions")
print("---")
print("NumPy:       ", np.__version__)
print("Pandas:      ", pd.__version__)
print("Delicatessen:", deli.__version__)
print("Cantilever:  ", cantilever.__version__)

Versions
---
NumPy:        1.25.2
Pandas:       1.4.1
Delicatessen: 3.2
Cantilever:   25.0a


In [2]:
d = pd.read_csv("data/actg_discrete.csv")
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1969 entries, 0 to 1968
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          1969 non-null   int64  
 1   y           1734 non-null   float64
 2   art         1969 non-null   float64
 3   study       1969 non-null   int64  
 4   male        1969 non-null   int64  
 5   black       1969 non-null   int64  
 6   idu         1969 non-null   int64  
 7   age         1969 non-null   int64  
 8   karnof_cat  1969 non-null   float64
 9   cd4         1969 non-null   int64  
dtypes: float64(3), int64(7)
memory usage: 154.0 KB


As seen here, there is some missing data for the outcome (`y`). However, there is no other missing data. The estimators described in this section can all handle missing outcome data (but not other types of missing data.

For ease, all the nuisance model specifications are provided here. Since both data sources were marginally randomized trials, the action model is an intercept-only model. However, baseline covariates could be added to potentially increase the efficiency (i.e., produce narrower confidence intervals). 

In [3]:
# Nuisance models: weights
action_model = "1"
sample_model = "male + age + black + idu + C(karnof_cat) + cd4"
censor_model = "C(art) + male + age + black + idu + C(karnof_cat) + cd4 + study"

# Nuisance models: outcome
outcome_model = "C(art, contr.treatment(1)) + male + age + black + idu + C(karnof_cat) + cd4"

## Inverse Probability Weighting

The first estimator considered is the inverse probability weighting (IPW) estimator. 

To start, we initialize the `BridgeIPW` class with our stacked data sets and designate the columns labels indicating the outcome, action, and sample. Here, we will also specify `verbose=True` so we can examine the estimated coefficients for our nuisance models.

In [4]:
bipw = BridgeIPW(data=d, outcome='y', action='art', sample='study', verbose=True)

The following functions take the specifications of each nuisance model for IPW and then fits that model using logistic regression. 

In [5]:
bipw.action_model(action_model)

Action Nuisance Model
--------------------------------------------------------------------
No. Observations:   1969        | Dependent Variable: art        
Model:              logistic    | Method:             lm         
                 coef  stderr  Z-value   LCL   UCL
_                                                 
S=1 : Intercept  0.69    0.07     9.32  0.55  0.84
S=0 : Intercept -0.00    0.06    -0.06 -0.12  0.11


In [6]:
bipw.sample_model(sample_model)

Sampling Nuisance Model
--------------------------------------------------------------------
No. Observations:   1969        | Dependent Variable: study      
Model:              logistic    | Method:             lm         
                      coef  stderr  Z-value   LCL   UCL
_                                                      
Intercept             5.56    0.57     9.73  4.44  6.68
C(karnof_cat)[T.1.0]  0.42    0.20     2.11  0.03  0.81
C(karnof_cat)[T.2.0]  0.55    0.34     1.61 -0.12  1.21
age                   0.03    0.01     2.17  0.00  0.05
black                -0.02    0.23    -0.10 -0.47  0.42
cd4                  -0.03    0.00   -17.67 -0.04 -0.03
idu                   0.49    0.27     1.82 -0.04  1.02
male                  0.04    0.27     0.16 -0.48  0.56


In [7]:
bipw.missing_model(censor_model)

Missingness Nuisance Model
--------------------------------------------------------------------
No. Observations:   1969        | Dependent Variable: _missing_y_var_
Model:              logistic    | Method:             lm         
                      coef  stderr  Z-value   LCL   UCL
_                                                      
Intercept             4.51    0.75     6.05  3.05  5.98
C(art)[T.1.0]         0.99    0.77     1.29 -0.52  2.51
C(art)[T.2.0]         1.12    0.79     1.42 -0.42  2.66
C(karnof_cat)[T.1.0] -0.19    0.16    -1.18 -0.52  0.13
C(karnof_cat)[T.2.0] -0.13    0.23    -0.59 -0.57  0.31
age                   0.03    0.01     3.78  0.02  0.05
black                -0.51    0.16    -3.16 -0.82 -0.19
cd4                  -0.00    0.00    -4.03 -0.01 -0.00
idu                   0.05    0.21     0.23 -0.36  0.45
male                  0.17    0.19     0.89 -0.21  0.54
study                -4.97    0.69    -7.15 -6.33 -3.61


After fitting each of the nuisance models, we can then estimate the 3-drug versus 1-drug comparison. The following function does this process.

In [8]:
bipw.estimate()

Before examining our results, it is good practice to first look at the diagnostics available for the corresponding estimator. The available diagnostics can all be run using the following function

In [9]:
bipw.diagnostics()

Weight Diagnostics
Inverse Probability of Treatment Weights
 * The overall 'Mean' column should be near 2
 * Action-specific 'Sum' columns should be approximately equal
--------------------------------------------------------------
study = 0
--------------------------------------------------------------
          art  art=0  art=1
_                          
Mean     2.00    3.0    1.5
SD       0.71    0.0    0.0
Min      1.50    3.0    1.5
P25      1.50    3.0    1.5
P50      1.50    3.0    1.5
P75      3.00    3.0    1.5
Max      3.00    3.0    1.5
Sum   1626.00  813.0  813.0
--------------------------------------------------------------
study = 1
--------------------------------------------------------------
         art   art=1   art=2
_                           
Mean     2.0     2.0     2.0
SD       0.0     0.0     0.0
Min      2.0     2.0     2.0
P25      2.0     2.0     2.0
P50      2.0     2.0     2.0
P75      2.0     2.0     2.0
Max      2.0     2.0     2.0
Sum   2312.0  1156

From the shared arm diagnostic, we can see there is a (statistically significant) non-zero difference. This result is indicative of some underlying assumption for the validity of the IPW estimator not being met. Looking through the weights, it should be apparent that there are rather extreme inverse odds of sampling weights. After a bit more investigation, you would likely find that these extreme weights are a result of a positivity violation. Specifically, the ACTG 320 and ACTG 175 had different, non-overlapping inclusion for baseline CD4. As baseline CD4 is a strong predictor of the outcome, this likely invalidates this fusion of the trials. 

For purposes of this tutorial, we will ignore this diagnostic and now look at the results for the comparison of interest

In [10]:
bipw.summary()

Estimator:        Inverse Probability Weighting - Hajek
--------------------------------------------------------------
No. Observations: 1969       | No. Input:        1969      
No. w/ Outcomes:  1734       | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            none      
Alpha:            0.05       | 
--------------------------------------------------------------
             Estimate    SE   LCL   UCL  P-value
                                                
Single-span     -0.05  0.05 -0.15  0.04     0.26
Multi-span      -0.20  0.05 -0.30 -0.10     0.00
Diagnostic       0.15  0.02  0.12  0.18     0.00
A=2,S=1          0.06  0.01  0.04  0.09      NaN
A=1,S=1          0.15  0.02  0.12  0.18      NaN
A=1,S=0          0.00  0.00 -0.00  0.00      NaN
A=0,S=0          0.12  0.05  0.03  0.21      NaN


This summary table provides the estimated means using the single-span and multi-span expressions. Here, different results are produced between the expressions (due to the non-zero difference between the mean for the shared 2-drug arm).

Given the positivity violation by baseline CD4 for the sampling model, we now restrict the data by baseline CD4 where there is greater overlap between the two trials

In [17]:
ds = d.loc[(d['cd4'] >= 100) & (d['cd4'] <= 300) ].copy()

In [18]:
bipw = BridgeIPW(data=ds, outcome='y', action='art', sample='study', verbose=False)
bipw.action_model(action_model)
bipw.sample_model(sample_model)
bipw.missing_model(censor_model)
bipw.estimate()

In [20]:
bipw.diagnostics()

Weight Diagnostics
Inverse Probability of Treatment Weights
 * The overall 'Mean' column should be near 2
 * Action-specific 'Sum' columns should be approximately equal
--------------------------------------------------------------
study = 0
--------------------------------------------------------------
         art   art=0   art=1
_                           
Mean    2.00    3.03    1.49
SD      0.72    0.00    0.00
Min     1.49    3.03    1.49
P25     1.49    3.03    1.49
P50     1.49    3.03    1.49
P75     3.03    3.03    1.49
Max     3.03    3.03    1.49
Sum   666.00  333.00  333.00
--------------------------------------------------------------
study = 1
--------------------------------------------------------------
         art   art=1   art=2
_                           
Mean    2.00    2.13    1.89
SD      0.12    0.00    0.00
Min     1.89    2.13    1.89
P25     1.89    2.13    1.89
P50     1.89    2.13    1.89
P75     2.13    2.13    1.89
Max     2.13    2.13    1.89
Sum   90

While restricting by CD4 improved the inverse odds of sampling weights and the shared arm diagnostic, they both still indicate there are substantively important differences. Therefore, the results from this fusion remain suspect (and one would likely conclude that this fusion has failed). 

For completeness, the 3-drug versus 1-drug comparison is made below.

In [19]:
bipw.summary()

Estimator:        Inverse Probability Weighting - Hajek
--------------------------------------------------------------
No. Observations: 784        | No. Input:        784       
No. w/ Outcomes:  670        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            none      
Alpha:            0.05       | 
--------------------------------------------------------------
             Estimate    SE   LCL   UCL  P-value
                                                
Single-span     -0.09  0.05 -0.18  0.00     0.05
Multi-span      -0.14  0.05 -0.24 -0.04     0.01
Diagnostic       0.05  0.02  0.02  0.08     0.00
A=2,S=1          0.02  0.01 -0.00  0.04      NaN
A=1,S=1          0.05  0.02  0.02  0.08      NaN
A=1,S=0          0.00  0.00 -0.00  0.00      NaN
A=0,S=0          0.11  0.05  0.02  0.20      NaN


Here, the single-span and multi-span are more similar than before. This is again due to the shrinking difference between the 2-drug arms.

## G-computation

An alternative estimator is bridge g-computation. Rather than estimating the weights, this approach relies on modeling the outcome process for each trial. Therefore, only a single model needs to be fit. 

One advantage of g-computation is that it can better handle positivity violations (but this is premised on a parametric model being correctly specified). 

To start, we initialize the `BridgeGComputation` class with our stacked data sets and designate the columns labels indicating the outcome, action, and sample. Using what we discovered with the IPW estimator, we use the baseline CD4 restricted data. Here, we will also specify `verbose=True` so we can examine the estimated coefficients for our nuisance model.

In [26]:
bgcomp = BridgeGComputation(data=ds, outcome='y', action='art', sample='study', verbose=True)

The next step fits the outcome nuisance model. Again, notice that this is the only nuisance model. Here, outcome models are fit for each trial separately. The coefficients from each model are indicated by the leading `S = s` indicator in the output table

In [27]:
bgcomp.outcome_model(outcome_model, model_type='logistic')

Outcome Nuisance Model
--------------------------------------------------------------------
No. Observations:   670         | Dependent Variable: y          
Model:              logistic    | Method:             lm         
                           coef  stderr  Z-value   LCL   UCL
_                                                           
S=1 : Intercept            1.80    2.52     0.71 -3.14  6.74
S=1 : C(art, contr.treatm  0.00    0.00     0.00 -0.00  0.00
S=1 : C(art, contr.treatm -1.21    0.64    -1.87 -2.47  0.06
S=1 : C(karnof_cat)[T.1.0 -0.82    0.72    -1.14 -2.23  0.59
S=1 : C(karnof_cat)[T.2.0  0.49    0.97     0.51 -1.42  2.40
S=1 : age                 -0.03    0.05    -0.58 -0.14  0.08
S=1 : black                0.47    0.67     0.70 -0.84  1.78
S=1 : cd4                 -0.02    0.01    -2.11 -0.04 -0.00
S=1 : idu                 -0.77    1.25    -0.62 -3.21  1.67
S=1 : male                -0.45    0.67    -0.67 -1.77  0.87
S=0 : Intercept           -5.00    2.04    -

After fitting the outcome nuisance model, we can then estimate the 3-drug versus 1-drug comparison. The following function does this process.

In [28]:
bgcomp.estimate()

As with the IPW estimator, we can also run some diagnostic procedures. Note that g-computation has different diagnostics for the nuisance model, but the diagnostic based on the shared arms is the same

In [29]:
bgcomp.diagnostics()

Outcome Regression Residuals
 * Skewed or large residual may indicate misspecification
--------------------------------------------------------------
      study=0  study=1
_                     
Mean   -0.000    0.000
SD      0.186    0.170
Min    -0.996   -0.988
P25     0.006    0.006
P50     0.010    0.017
P75     0.052    0.041
Max     0.413    0.295

Shared Arm Diagnostic
No. Observations: 784        | No. Input:        784       
No. w/ Outcomes:  670        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
--------------------------------------------------------------
No. Observations: 784        | No. Input:        784       
No. w/ Outcomes:  670        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
            E

Here, the shared arm diagnostic looks a bit better than the IPW estimator. This aligns with what we might expect regarding how g-computation can better handle positivity violations. However, these results still suggest that there might be issues with this fusion given the relatively small P-value that remains for the diagnostic. So one should still be a bit skeptical of these results. 

However, the 3-drug versus 1-drug comparison results are shown here to illustrate the `summary` function

In [24]:
bgcomp.summary()

Estimator:        Parametric G-computation
--------------------------------------------------------------
No. Observations: 784        | No. Input:        784       
No. w/ Outcomes:  670        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
--------------------------------------------------------------
             Estimate    SE   LCL   UCL  P-value
                                                
Single-span     -0.15  0.09 -0.32  0.02     0.08
Multi-span      -0.18  0.09 -0.35 -0.02     0.03
Diagnostic       0.03  0.02 -0.00  0.07     0.08
A=2,S=1          0.02  0.01 -0.00  0.03      NaN
A=1,S=1          0.05  0.02  0.02  0.08      NaN
A=1,S=0          0.02  0.01 -0.00  0.04      NaN
A=0,S=0          0.17  0.09 -0.00  0.34      NaN


## Augmented Inverse Probability Weighting

The final estimator is the augmented inverse probability weighting (AIPW) estimator. This estimator can be viewed as a combination of the previous IPW and g-computation estimators.

Here, the weighted-regression implementation of AIPW is used. 

To start, we initialize the `BridgeAIPW` class with our stacked CD4-restricted data sets and designate the columns labels indicating the outcome, action, and sample. Here, we will specify `verbose=False` since the nuisance model coefficients will be the same for most nuisance models.

In [38]:
baipw = BridgeAIPW(data=ds, outcome='y', action='art', sample='study', verbose=False)

In [39]:
baipw.action_model(action_model)
baipw.sample_model(sample_model)
baipw.missing_model(censor_model)
baipw.outcome_model(outcome_model, model_type='logistic')

In [40]:
baipw.estimate()

Diagnostics for the AIPW estimator can be output via the `diagnostics` function. As these diagnostics are a combination of the previous IPW and g-computation diagnostics, we do not output them here.

The following are the summary results for the AIPW estimator

In [41]:
baipw.summary()

Estimator:        Augmented Inverse Probability Weighting
--------------------------------------------------------------
No. Observations: 784        | No. Input:        784       
No. w/ Outcomes:  670        | Outcome:          y         
Action:           art        | Sample:           study     
Outcome type:     Binary     | Model:            logistic  
Alpha:            0.05       | 
--------------------------------------------------------------
             Estimate    SE   LCL   UCL  P-value
                                                
Single-span     -0.08  0.04 -0.17  0.00     0.06
Multi-span      -0.13  0.05 -0.23 -0.04     0.00
Diagnostic       0.05  0.02  0.02  0.08     0.00
A=2,S=1          0.02  0.01 -0.00  0.04      NaN
A=1,S=1          0.05  0.02  0.02  0.08      NaN
A=1,S=0          0.00  0.00 -0.00  0.00      NaN
A=0,S=0          0.10  0.04  0.02  0.19      NaN


Again, a substantial difference for the shared arm diagnostic is observed.

References 

* Hammer SM et al. (1996). A trial comparing nucleoside monotherapy with combination therapy in HIV-infected adults with CD4 cell counts from 200 to 500 per cubic millimeter. *New England Journal of Medicine*, 335(15):1081-1090.
* Hammer SM, et al. (1997). A controlled trial of two nucleoside analogues plus indinavir in persons with human immunodeficiency virus infection and CD4 cell counts of 200 per cubic millimeter or less. *New England Journal of Medicine*, 337(11):725-733.
* Shook-Sa BE, Zivich PN, Rosin SP, Edwards JK, Adimora AA, Hudgens MG, Cole SR. (2024). Fusing Trial Data for Treatment Comparisons: Single versus Multi-Span Bridging. *Statistics in Medicine*, 43(4):793-815.
